In [ ]:
!pip uninstall -y pillow
!pip install pillow==10.3.0

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes pillow pandas trl huggingface_hub

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path
from typing import Dict, List, Optional
from dataclasses import dataclass
from tqdm.auto import tqdm

from datasets import Dataset, DatasetDict, Features, Value, Image as ImageFeature
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import HfApi, login

In [ ]:
!wget -nc -q https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip
!wget -nc -q https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_GroundTruth.csv

In [ ]:
!unzip -q -o /kaggle/working/ISIC_2019_Training_Input.zip

In [ ]:
df = pd.read_csv("ISIC_2019_Training_GroundTruth.csv")
print(f"✅ Loaded {len(df)} samples")
print(f"Columns: {df.columns.tolist()}")

✅ Loaded 25331 samples
Columns: ['image', 'MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']


In [ ]:
LABEL_COLUMNS = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']
LABEL_NAMES = {
    'MEL': 'Melanoma',
    'NV': 'Melanocytic Nevus',
    'BCC': 'Basal Cell Carcinoma',
    'AK': 'Actinic Keratosis',
    'BKL': 'Benign Keratosis',
    'DF': 'Dermatofibroma',
    'VASC': 'Vascular Lesion',
    'SCC': 'Squamous Cell Carcinoma',
    'UNK': 'Unknown'
}


In [ ]:
def get_diagnosis(row):
    """Extract diagnosis from one-hot encoded columns"""
    for col in LABEL_COLUMNS:
        if row[col] == 1.0:
            return LABEL_NAMES[col]
    return "Unknown"

def create_training_prompt(diagnosis: str, detailed: bool = True) -> str:
    """Create prompts for fine-tuning"""
    if detailed:
        return f"""Describe the dermatological features of this lesion focusing on:
- Asymmetry
- Border characteristics
- Color variations
- Diameter and texture
- Any distinctive patterns

Provide a clinical assessment."""
    else:
        return "Describe this skin lesion in clinical terms."

def create_training_response(diagnosis: str) -> str:
    """Create detailed clinical responses based on diagnosis"""

    responses = {
        'Melanoma': """This lesion exhibits characteristics concerning for melanoma:
- Asymmetry: The lesion shows irregular asymmetric growth
- Border: Borders are irregular and poorly defined
- Color: Multiple colors present including brown, black, and possible red/blue tones
- Diameter: Appears greater than 6mm
- Evolution: Clinical correlation needed for changes over time

Assessment: Highly suspicious for melanoma. Immediate dermatology referral and biopsy recommended. The ABCDE criteria are met, warranting urgent evaluation.""",

        'Melanocytic Nevus': """This lesion demonstrates features consistent with a benign melanocytic nevus:
- Symmetry: Relatively symmetric appearance
- Border: Well-defined, regular borders
- Color: Uniform brown coloration
- Diameter: Typically less than 6mm
- Surface: Smooth texture

Assessment: Consistent with benign nevus. Routine monitoring recommended. Patient should be educated on ABCDE criteria for melanoma surveillance.""",

        'Basal Cell Carcinoma': """This lesion shows features suggestive of basal cell carcinoma:
- Appearance: Pearly or waxy quality visible
- Border: Rolled borders with possible central ulceration
- Color: Typically pink to flesh-colored, may have telangiectasias
- Texture: May appear translucent with visible blood vessels

Assessment: Suspicious for basal cell carcinoma. Biopsy recommended for histopathological confirmation. Treatment options include surgical excision, Mohs surgery, or topical therapy depending on subtype and location.""",

        'Actinic Keratosis': """This lesion demonstrates features of actinic keratosis:
- Texture: Rough, scaly, or crusty surface
- Color: Pink, red, or brown discoloration
- Location: Sun-exposed areas (face, scalp, hands, forearms)
- Size: Usually small, less than 1 cm
- Palpation: Better felt than seen - sandpaper-like texture

Assessment: Consistent with actinic keratosis, a precancerous condition. Treatment recommended to prevent progression to squamous cell carcinoma. Options include cryotherapy, topical 5-FU, imiquimod, or photodynamic therapy.""",

        'Benign Keratosis': """This lesion shows characteristics of seborrheic keratosis:
- Appearance: "Stuck-on" waxy or warty surface
- Color: Tan, brown, or black
- Texture: Rough, slightly raised surface
- Border: Well-demarcated edges
- Pattern: May show horn cysts or milia-like cysts

Assessment: Consistent with benign seborrheic keratosis. No malignant features observed. Treatment not medically necessary but can be performed for cosmetic reasons or if symptomatic (irritation, bleeding).""",

        'Dermatofibroma': """This lesion exhibits features consistent with dermatofibroma:
- Palpation: Firm, button-like nodule
- Color: Brown to red-brown, may have peripheral hyperpigmentation
- Size: Typically 5-10mm in diameter
- Sign: Dimple sign positive (dimpling when pinched)
- Location: Most common on lower extremities

Assessment: Consistent with benign dermatofibroma. No treatment necessary unless symptomatic or for cosmetic concerns. Surgical excision possible but may result in more prominent scarring.""",

        'Vascular Lesion': """This lesion demonstrates vascular characteristics:
- Color: Red to purple coloration
- Blanching: May blanch with pressure
- Pattern: May show telangiectatic or angiomatous patterns
- Border: Variable, can be well-defined or diffuse
- Surface: Can be flat or raised

Assessment: Consistent with vascular lesion (cherry angioma, spider angioma, or other benign vascular proliferation). Generally benign and no treatment required unless symptomatic or for cosmetic purposes. Laser therapy effective if treatment desired.""",

        'Squamous Cell Carcinoma': """This lesion shows features concerning for squamous cell carcinoma:
- Texture: Scaly, crusty, or ulcerated surface
- Growth: Rapidly growing firm nodule or plaque
- Color: Red, pink, or flesh-colored
- Border: Irregular margins, may be indurated
- Location: Sun-exposed areas, sites of chronic inflammation or scars

Assessment: Highly suspicious for squamous cell carcinoma. Urgent biopsy required. Risk of metastasis exists, particularly with lesions >2cm, poor differentiation, or perineural invasion. Prompt surgical excision or Mohs surgery recommended."""
    }

    return responses.get(diagnosis, f"Clinical assessment needed for {diagnosis}.")

In [ ]:
image_dir = Path("/kaggle/working/ISIC_2019_Training_Input")

In [ ]:
data_list = []
missing_images = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing images"):
    image_path = image_dir / f"{row['image']}.jpg"

    if not image_path.exists():
        missing_images += 1
        continue

    diagnosis = get_diagnosis(row)
    if diagnosis == "Unknown":
        continue

    prompt = create_training_prompt(diagnosis, detailed=True)
    response = create_training_response(diagnosis)

    data_list.append({
        'image': str(image_path),
        'diagnosis': diagnosis,
        'prompt': prompt,
        'response': response,
        'image_id': row['image']
    })

print(f"Prepared {len(data_list)} samples")
print(f"Missing {missing_images} images")


In [ ]:
from sklearn.model_selection import train_test_split

train_data, eval_data = train_test_split(data_list, test_size=0.1, random_state=42, stratify=[d['diagnosis'] for d in data_list])

print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")

📊 Train: 22797, Eval: 2534


In [ ]:
!rm /kaggle/working/ISIC_2019_Training_Input.zip

In [ ]:
train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

In [ ]:
dataset_dict = DatasetDict({
    'train': train_dataset,
    'eval': eval_dataset
})

print(dataset_dict)
print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")

DatasetDict({
    train: Dataset({
        features: ['image', 'diagnosis', 'prompt', 'response', 'image_id'],
        num_rows: 22797
    })
    eval: Dataset({
        features: ['image', 'diagnosis', 'prompt', 'response', 'image_id'],
        num_rows: 2534
    })
})
Train samples: 22797
Eval samples: 2534


In [ ]:
MODEL_ID = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

if "google.colab" in sys.modules:
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')
        login(token=hf_token)
    except:
        print("HF_TOKEN not found in Colab secrets. Please set it up or login manually.")
        login()
else:
    login()

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)



processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,  # Alpha parameter
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # Attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="FEATURE_EXTRACTION"
)


In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 11,898,880 || all params: 4,311,978,352 || trainable%: 0.2759


In [ ]:
@dataclass
class MedGemmaDataCollator:
    processor: AutoProcessor

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        processed_samples = []

        for f in features:
            if isinstance(f['image'], str):
                image = Image.open(f['image']).convert('RGB')
            else:
                image = f['image']

            prompt = f['prompt']
            response = f['response']

            conversation = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": prompt}
                    ]
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": response}
                    ]
                }
            ]

            full_text = self.processor.apply_chat_template(
                conversation,
                tokenize=False,
                add_generation_prompt=False
            )
            processed = self.processor(
                text=full_text,
                images=image,
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=1024
            )

            processed_samples.append(processed)

        batch = {}

        keys = processed_samples[0].keys()

        for key in keys:
            if key == 'pixel_values':
                batch[key] = torch.cat([s[key] for s in processed_samples], dim=0)
            else:
                tensors = [s[key].squeeze(0) for s in processed_samples]
                batch[key] = torch.nn.utils.rnn.pad_sequence(
                    tensors,
                    batch_first=True,
                    padding_value=self.processor.tokenizer.pad_token_id
                )

        batch["labels"] = batch["input_ids"].clone()

        batch["labels"][batch["labels"] == self.processor.tokenizer.pad_token_id] = -100

        return batch

data_collator = MedGemmaDataCollator(processor=processor)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./medgemma-dermatology-finetuned",
    num_train_epochs=1,
    eval_strategy="steps",
    eval_steps=100,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=20,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False
)

In [ ]:
class PrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"Step {state.global_step}: {logs}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    callbacks=[PrinterCallback()]
)



In [ ]:
trainer.train()

In [ ]:
output_dir = "./medgemma-dermatology-finetuned-final"
trainer.model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

['./medgemma-dermatology-finetuned-final/processor_config.json']

In [ ]:
HF_USERNAME = "ayyuce"
MODEL_NAME = "medgemma-dermatology-isic2019-full-1ep"
repo_id = f"{HF_USERNAME}/{MODEL_NAME}"

medgemma-dermatology-isic2019 medgemma-dermatology-isic2019
